# DareDevil — Paper-Ready Evaluation Notebook

Loads trained models from `faizmoazam/daredevil-trained-models` and evaluates on the
**held-out test set** (`faizmoazam/held-out`) — data never seen during training or validation.

Produces:
1. Dataset section figures (class distribution, unsupervised explanation)
2. Val results table (from cached pickled results — no retraining)
3. Test results table on held-out data (all 4 models)
4. Per-class breakdown (person / other_object separate)
5. Per-direction F1 grouped bar (all models)
6. Feature Fusion only: person vs other_object per direction
7. Shuffle robustness table on held-out data (blind / classifier / oracle)
8. Latency note (text only, no figure)

All figures use a black/white-compatible style suitable for IEEE paper printing.


## Section 1 — Imports & config

In [25]:
import matplotlib
matplotlib.use("Agg")
import copy, json, pickle, random, re, time, warnings
from datetime import datetime
from itertools import permutations
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, confusion_matrix
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda


In [26]:
import matplotlib
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family":       "serif",
    "font.serif":        ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size":         13,
    "axes.titlesize":    14,
    "axes.labelsize":    13,
    "xtick.labelsize":   12,
    "ytick.labelsize":   12,
    "legend.fontsize":   11,
    "figure.dpi":        150,
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "grid.linestyle":    "--",
    "grid.color":        "#AAAAAA",
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

In [27]:
!pip install ultralytics

In [28]:
# ---- paper figure style: Times New Roman, pure grayscale for IEEE paper ----
plt.rcParams.update({
    "font.family":       "serif",
    "font.serif":        ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size":         13,
    "axes.titlesize":    14,
    "axes.labelsize":    13,
    "xtick.labelsize":   12,
    "ytick.labelsize":   12,
    "legend.fontsize":   11,
    "figure.dpi":        150,
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "grid.linestyle":    "--",
    "grid.color":        "#AAAAAA",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "text.color":        "black",
    "axes.edgecolor":    "black",
    "axes.labelcolor":   "black",
    "xtick.color":       "black",
    "ytick.color":       "black",
})

OUT_DIR = Path("/kaggle/working/paper_figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def savefig(name, tight=True):
    path = OUT_DIR / f"{name}.pdf"
    if tight:
        plt.tight_layout()
    plt.savefig(path, bbox_inches="tight", facecolor="white")
    plt.savefig(OUT_DIR / f"{name}.png", dpi=200, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"  saved -> {path}")


print("Style configured — Times New Roman, grayscale. Output dir:", OUT_DIR)


Style configured — Times New Roman, grayscale. Output dir: /kaggle/working/paper_figures


In [29]:
import matplotlib
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family":       "serif",
    "font.serif":        ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size":         13,
    "axes.titlesize":    14,
    "axes.labelsize":    13,
    "xtick.labelsize":   12,
    "ytick.labelsize":   12,
    "legend.fontsize":   11,
    "figure.dpi":        150,
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "grid.linestyle":    "--",
    "grid.color":        "#AAAAAA",
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

In [30]:
# ---- paths ----
# Training data paths (only used for label-generation infrastructure reuse)
DATA_ROOT         = Path("/kaggle/input/datasets/faizmoazam/full-stitched-and-unstitched/all_runs - Copy")
HOMOGRAPHY_DIR    = Path("/kaggle/input/datasets/faizmoazam/thermal-homography-mats/thermal_homography_mats 2/thermal_homography_mats")
RGB_HOMOGRAPHY_DIR = Path("/kaggle/input/datasets/faizmoazam/rgb-homography/rgb homography final/rgb homography final")

# Held-out test set -- same structure as training parts (thermal-g*/rgb-g*/sync-g*)
HELD_OUT_ROOT     = Path("/kaggle/input/datasets/faizmoazam/new-mini-dataset/latest-g20260616T123146Z")

# Models dataset: saved_models/ and results/ are nested inside daredevil_models_and_results/
MODELS_DIR        = Path("/kaggle/input/datasets/faizmoazam/daredevil-trained-models/daredevil_models_and_results/saved_models")
RESULTS_DIR       = Path("/kaggle/input/datasets/faizmoazam/daredevil-trained-models/daredevil_models_and_results/results")
YOLO_WEIGHTS      = "yolov8x.pt"

# ---- rig geometry ----
SENSOR_W, SENSOR_H = 32, 24
ACTIVE_CAMERAS      = [2, 5, 4]           # left, middle, right
CAM_ORDER_LEFT_TO_RIGHT = ACTIVE_CAMERAS
DIRECTIONS          = ["left", "middle", "right"]
CAM_TO_DIRECTION    = dict(zip(ACTIVE_CAMERAS, DIRECTIONS))
CLUSTER_BUILD_ORDER = [4, 5, 2]
NUM_CAMS            = len(ACTIVE_CAMERAS)
ALL_CAMS            = [3, 2, 5, 4, 6]

CLASS_NAMES = ["no_object", "person", "other_object"]
NUM_CLASSES = 3
COCO_PERSON_CLASS_ID = 0
MAX_RGB_GAP_MS = 150

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

SMOKE_TEST = False   # set True to test on first 60 held-out samples only
SMOKE_N    = 60


## Section 2 — IO utilities

In [31]:
import re as _re
CAPTURE_RE = _re.compile(r"^data_capture_(\d+)_(\d+)_(\d+)\.json$")
CAM_NAME_RE = _re.compile(r"cam(\d+)")
FLIP_THERMAL_HORIZONTALLY = True


def load_capture(path):
    with open(path) as f:
        text = f.read()
    text = _re.sub(r"-nan\b", "NaN", text)
    return json.loads(text)


def list_captures(folder, cam_id):
    folder = Path(folder)
    captures = []
    for f in folder.iterdir():
        m = CAPTURE_RE.match(f.name)
        if m and int(m.group(1)) == cam_id:
            captures.append((int(m.group(2)), f))
    captures.sort(key=lambda x: x[0])
    return [p for _, p in captures]


def build_camera_sequence(folder, cam_id):
    paths = list_captures(folder, cam_id)
    sequence = []
    buffer = np.zeros(SENSOR_W * SENSOR_H, dtype=np.float64)
    for path in paths:
        data = load_capture(path)
        raw = np.array(data["data"], dtype=np.float64)
        mask = (raw != 0) & ~np.isnan(raw)
        buffer[mask] = raw[mask]
        grid = buffer.copy().reshape(SENSOR_H, SENSOR_W)
        if FLIP_THERMAL_HORIZONTALLY:
            grid = np.ascontiguousarray(grid[:, ::-1])
        sequence.append((data["frame_index"], data["unix_time_ms"], grid))
    return sequence


def load_homography_json(path):
    with open(path) as f:
        return json.load(f)


def discover_homography_files(root):
    root = Path(root)
    entries = []
    for path in sorted(root.rglob("*_homography.json")):
        data = load_homography_json(path)
        if data.get("cam_a") == "rgb":
            continue
        a_m = CAM_NAME_RE.match(data.get("cam_a", ""))
        b_m = CAM_NAME_RE.match(data.get("cam_b", ""))
        if not (a_m and b_m):
            continue
        entries.append({"cam_a": int(a_m.group(1)), "cam_b": int(b_m.group(1)),
                        "H": np.array(data["H"], dtype=np.float64)})
    return entries


def discover_rgb_homographies(root):
    root = Path(root)
    calibrations = {}
    for path in sorted(root.rglob("rgb*_homography.json")):
        data = load_homography_json(path)
        if data.get("cam_a") != "rgb":
            continue
        m = CAM_NAME_RE.match(data.get("cam_b", ""))
        if not m:
            continue
        cam_id = int(m.group(1))
        quad_a = np.array(data["quad_a"], dtype=np.float64)
        calibrations[cam_id] = {
            "H": np.array(data["H"], dtype=np.float64),
            "quad_a": quad_a,
            "quad_a_centroid": quad_a.mean(axis=0),
        }
    return calibrations


def load_sync_table(csv_path):
    return pd.read_csv(csv_path).to_dict("records")


def rgb_gap_ok(row, max_gap_ms=MAX_RGB_GAP_MS):
    gap = row.get("rgb_gap_ms", "-1")
    files = row.get("rgb_files", "")
    if not files:
        return False
    try:
        return 0 <= float(gap) <= max_gap_ms
    except ValueError:
        return False


## Section 3 — Panorama geometry

In [32]:
def _transform_corners(w, h, T):
    corners = np.array([[0,0],[w-1,0],[w-1,h-1],[0,h-1]], dtype=np.float64)
    ch = np.hstack([corners, np.ones((4,1))])
    t = (T @ ch.T).T
    return t[:,:2] / t[:,2:3]


def compute_canvas_bounds(shapes, transforms):
    all_pts = np.vstack([_transform_corners(w, h, T) for (h,w),T in zip(shapes, transforms)])
    min_x, min_y = np.floor(all_pts.min(0)).astype(int)
    max_x, max_y = np.ceil(all_pts.max(0)).astype(int)
    return min_x, min_y, max_x, max_y


def build_clusters(homography_entries, cam_ids):
    graph = {c: [] for c in cam_ids}
    for e in homography_entries:
        a, b, H = e["cam_a"], e["cam_b"], e["H"]
        if a not in graph or b not in graph:
            continue
        graph[a].append((b, np.linalg.inv(H)))
        graph[b].append((a, H))
    visited, clusters = set(), []
    for root in cam_ids:
        if root in visited:
            continue
        visited.add(root)
        cams, transforms, t2r = [root], [np.eye(3)], {root: np.eye(3)}
        queue = [root]
        while queue:
            cur = queue.pop(0)
            for nb, T in graph[cur]:
                if nb in visited: continue
                visited.add(nb)
                t2r[nb] = t2r[cur] @ T
                cams.append(nb); transforms.append(t2r[nb]); queue.append(nb)
        clusters.append({"cams": cams, "transforms": transforms})
    return clusters


homography_entries = discover_homography_files(HOMOGRAPHY_DIR)
clusters = build_clusters(homography_entries, CLUSTER_BUILD_ORDER)
main_cluster = next(
    (c for c in clusters if set(c["cams"]) == set(ACTIVE_CAMERAS)), None)
if main_cluster is None:
    raise RuntimeError(f"Could not find connected cluster for {ACTIVE_CAMERAS}")

PANO_MIN_X, PANO_MIN_Y, PANO_MAX_X, PANO_MAX_Y = compute_canvas_bounds(
    [(SENSOR_H, SENSOR_W)] * len(main_cluster["cams"]), main_cluster["transforms"])
PANO_W = PANO_MAX_X - PANO_MIN_X
PANO_H = PANO_MAX_Y - PANO_MIN_Y
PANO_TRANSLATION = np.array([[1,0,-PANO_MIN_X],[0,1,-PANO_MIN_Y],[0,0,1]], dtype=np.float64)

CAM_FOOTPRINTS, CAM_TRANSFORMS = {}, {}
for cam_id, T in zip(main_cluster["cams"], main_cluster["transforms"]):
    CAM_TRANSFORMS[cam_id] = T
    full_T = PANO_TRANSLATION @ T
    corners = _transform_corners(SENSOR_W, SENSOR_H, full_T)
    x1,y1 = corners[:,0].min(), corners[:,1].min()
    x2,y2 = corners[:,0].max(), corners[:,1].max()
    CAM_FOOTPRINTS[cam_id] = {"polygon": corners, "bbox": (x1,y1,x2,y2)}

RGB_CALIBRATIONS = discover_rgb_homographies(RGB_HOMOGRAPHY_DIR)
CAM_IDS_WITH_RGB_OVERLAP = set(RGB_CALIBRATIONS) & set(ACTIVE_CAMERAS)
PANO_T_FOR_CAM = {c: PANO_TRANSLATION @ T for c, T in CAM_TRANSFORMS.items()}
print("cluster cams:", main_cluster["cams"])
print("RGB overlap:", sorted(CAM_IDS_WITH_RGB_OVERLAP))


cluster cams: [4, 5, 2]
RGB overlap: [2, 4, 5]


## Section 4 — Held-out data discovery & label generation

Discovers the held-out capture session and runs YOLO label generation on it.
This data was **never seen** during training or validation.

In [33]:
def _find_one(root, prefix):
    matches = sorted(Path(root).glob(f"{prefix}-g*"))
    if not matches:
        raise FileNotFoundError(f"No '{prefix}-g*' under {root}")
    return matches[0]


def discover_held_out_parts(held_out_root):
    '''Discovers all part* folders under held_out_root, or treats held_out_root itself
    as a single capture session if it directly contains thermal-g*/rgb-g* folders.'''
    root = Path(held_out_root)
    # Check if it's structured as part* subfolders
    part_dirs = sorted(root.glob("part*"))
    if part_dirs:
        parts = {}
        for p in part_dirs:
            try:
                parts[p.name] = {
                    "thermal_dir": _find_one(p, "thermal"),
                    "rgb_dir": _find_one(p, "rgb"),
                    "sync_csv": _find_one(p, "sync") / "sync.csv",
                }
            except FileNotFoundError:
                pass
        return parts
    # Otherwise treat root itself as a single session
    try:
        return {"held_out": {
            "thermal_dir": _find_one(root, "thermal"),
            "rgb_dir": _find_one(root, "rgb"),
            "sync_csv": _find_one(root, "sync") / "sync.csv",
        }}
    except FileNotFoundError:
        # try one level down (e.g. held_out/latest-g20260616T123146Z/)
        subdirs = [d for d in root.iterdir() if d.is_dir() and not d.name.startswith(".")]
        if len(subdirs) == 1:
            sub = subdirs[0]
            return {"held_out": {
                "thermal_dir": _find_one(sub, "thermal"),
                "rgb_dir": _find_one(sub, "rgb"),
                "sync_csv": _find_one(sub, "sync") / "sync.csv",
            }}
        raise


HELD_OUT_PARTS = discover_held_out_parts(HELD_OUT_ROOT)
print(f"Held-out parts discovered: {list(HELD_OUT_PARTS.keys())}")
for name, paths in HELD_OUT_PARTS.items():
    print(f"  {name}: thermal={paths['thermal_dir'].name}  "
          f"sync_csv={paths['sync_csv'].exists()}")


Held-out parts discovered: ['held_out']
  held_out: thermal=thermal-g20260616T123146Z  sync_csv=True


In [34]:
# ---- load held-out sync rows ----
held_out_rows = []
for part_name, paths in HELD_OUT_PARTS.items():
    rows = load_sync_table(paths["sync_csv"])
    for i, row in enumerate(rows):
        row["_part"] = part_name
        row.setdefault("anchor_index", i)
    held_out_rows.extend(rows)

if SMOKE_TEST:
    held_out_rows = held_out_rows[:SMOKE_N]
    print(f"SMOKE TEST: using first {SMOKE_N} held-out rows")

print(f"Total held-out rows: {len(held_out_rows)}")


Total held-out rows: 988


In [35]:
# ---- YOLO label generation (same as training pipeline) ----
from ultralytics import YOLO
from matplotlib.path import Path as MplPath

_yolo_model = YOLO(YOLO_WEIGHTS)

def _nearest_calibration(box_xyxy):
    x1,y1,x2,y2 = box_xyxy
    cx = np.array([(x1+x2)/2, (y1+y2)/2])
    best_cam, best_dist = None, np.inf
    for cam_id, calib in RGB_CALIBRATIONS.items():
        d = np.linalg.norm(cx - calib["quad_a_centroid"])
        if d < best_dist:
            best_cam, best_dist = cam_id, d
    return best_cam


def _project_rgb_box(box_xyxy, via_cam_id):
    x1,y1,x2,y2 = box_xyxy
    corners = np.array([[x1,y1],[x2,y1],[x2,y2],[x1,y2]], dtype=np.float64)
    ch = np.hstack([corners, np.ones((4,1))])
    combined = PANO_T_FOR_CAM[via_cam_id] @ RGB_CALIBRATIONS[via_cam_id]["H"]
    out = (combined @ ch.T).T
    return out[:,:2] / out[:,2:3]


def generate_labels_for_frame(rgb_path, cam_ids=None):
    cam_ids = cam_ids or sorted(CAM_IDS_WITH_RGB_OVERLAP)
    img = cv2.imread(str(rgb_path))
    if img is None:
        return None
    results = _yolo_model(img, verbose=False)[0]
    has_person = {c: False for c in cam_ids}
    has_other  = {c: False for c in cam_ids}
    for box, cls in zip(results.boxes.xyxy.cpu().numpy(), results.boxes.cls.cpu().numpy()):
        via_cam = _nearest_calibration(box)
        pano_corners = _project_rgb_box(box, via_cam)
        cx, cy = pano_corners[:,0].mean(), pano_corners[:,1].mean()
        is_person = int(cls) == COCO_PERSON_CLASS_ID
        for cam_id in cam_ids:
            poly = MplPath(CAM_FOOTPRINTS[cam_id]["polygon"])
            pts = list(map(tuple, pano_corners)) + [(cx, cy)]
            if any(poly.contains_point(p) for p in pts):
                if is_person: has_person[cam_id] = True
                else: has_other[cam_id] = True
    labels = {}
    for cam_id in cam_ids:
        if has_person[cam_id]: labels[cam_id] = 1
        elif has_other[cam_id]: labels[cam_id] = 2
        else: labels[cam_id] = 0
    return labels


_label_cache = {}

def get_held_out_label(row):
    key = (row["_part"], int(row["anchor_index"]))
    if key in _label_cache:
        return _label_cache[key]
    part = row["_part"]
    paths = HELD_OUT_PARTS[part]
    class_vec = np.zeros(NUM_CAMS, dtype=np.int64)
    mask_vec  = np.zeros(NUM_CAMS, dtype=np.float32)
    if rgb_gap_ok(row):
        result = generate_labels_for_frame(paths["rgb_dir"] / row["rgb_files"])
        if result is not None:
            for i, cam_id in enumerate(CAM_ORDER_LEFT_TO_RIGHT):
                if cam_id in CAM_IDS_WITH_RGB_OVERLAP:
                    class_vec[i] = result[cam_id]
                    mask_vec[i]  = 1.0
    _label_cache[key] = (class_vec, mask_vec)
    return class_vec, mask_vec


print("Running YOLO label generation on held-out data...")
for row in held_out_rows:
    get_held_out_label(row)
print(f"Done. {len(_label_cache)} labels cached.")


Running YOLO label generation on held-out data...
Done. 988 labels cached.


## Section 5 — Dataset figures

Class distribution across left/middle/right, suitable for the paper's dataset section.
Frames are described as "capture sequences" — no mention of "train/val split" since
these are dataset-level statistics.

In [36]:
def compute_direction_counts(rows, label_fn):
    counts = {d: {0: 0, 1: 0, 2: 0, "unsupervised": 0} for d in DIRECTIONS}
    for row in rows:
        class_vec, mask_vec = label_fn(row)
        for i, cam_id in enumerate(CAM_ORDER_LEFT_TO_RIGHT):
            d = CAM_TO_DIRECTION[cam_id]
            if mask_vec[i] > 0:
                counts[d][int(class_vec[i])] += 1
            else:
                counts[d]["unsupervised"] += 1
    return counts


# Load val/train distribution from the cached pkl results (avoids re-running YOLO on training data)
val_result_b1 = pickle.load(open(RESULTS_DIR / "baseline1_stitched_result.pkl", "rb"))

# Rebuild training-data class counts from the saved class counts inside the result pkls
# (they're stored as part of save_eval_result's payload)
# If not available, use held-out counts as a proxy for the figure
held_out_counts = compute_direction_counts(held_out_rows, get_held_out_label)
print("Held-out class counts per direction:")
for d in DIRECTIONS:
    print(f"  {d}: {held_out_counts[d]}")


Held-out class counts per direction:
  left: {0: 638, 1: 18, 2: 328, 'unsupervised': 4}
  middle: {0: 168, 1: 161, 2: 655, 'unsupervised': 4}
  right: {0: 552, 1: 26, 2: 406, 'unsupervised': 4}


In [37]:
def plot_class_distribution(counts, title, filename):
    '''Stacked bar chart: class distribution by camera direction.
    Black/white compatible via hatching.'''
    directions = DIRECTIONS
    no_obj  = [counts[d][0] for d in directions]
    person  = [counts[d][1] for d in directions]
    other   = [counts[d][2] for d in directions]
    unsuper = [counts[d]["unsupervised"] for d in directions]

    x = np.arange(len(directions))
    fig, ax = plt.subplots(figsize=(6, 4.5))

    b1 = ax.bar(x, no_obj,  label="No Object",    color="white",   edgecolor="black", hatch="",    linewidth=1.2)
    b2 = ax.bar(x, person,  bottom=no_obj,
                label="Person",       color="#888888", edgecolor="black", hatch="///",  linewidth=1.2)
    b3 = ax.bar(x, other,
                bottom=[a+b for a,b in zip(no_obj, person)],
                label="Other Object", color="#444444", edgecolor="black", hatch="...", linewidth=1.2)
    unsuper_bottom = [a+b+c for a,b,c in zip(no_obj, person, other)]
    b4 = ax.bar(x, unsuper, bottom=unsuper_bottom,
                label="No RGB Label\n(Unsupervised)", color="#CCCCCC", edgecolor="black",
                hatch="---", linewidth=1.2)

    ax.set_xticks(x)
    ax.set_xticklabels(["Left", "Middle", "Right"], fontsize=13)
    ax.set_xlabel("Camera Direction", fontsize=13)
    ax.set_ylabel("Number of Frames", fontsize=13)
    ax.set_title(title, fontsize=14, pad=10)
    ax.legend(loc="upper right", framealpha=0.9)
    savefig(filename)


plot_class_distribution(held_out_counts,
    "Class distribution by camera direction\n(held-out test set)",
    "dataset_class_distribution_heldout")


  saved -> /kaggle/working/paper_figures/dataset_class_distribution_heldout.pdf


In [38]:
def plot_person_vs_other_rate(counts, title, filename):
    '''Per-direction person-rate and other-object-rate as grouped bars.'''
    directions = DIRECTIONS
    supervised = [counts[d][0] + counts[d][1] + counts[d][2] for d in directions]
    person_rate = [counts[d][1] / max(s, 1) for d, s in zip(directions, supervised)]
    other_rate  = [counts[d][2] / max(s, 1) for d, s in zip(directions, supervised)]
    noobj_rate  = [counts[d][0] / max(s, 1) for d, s in zip(directions, supervised)]

    x = np.arange(len(directions))
    w = 0.25
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(x - w, noobj_rate,  w, label="No Object",    color="white",   edgecolor="black", hatch="")
    ax.bar(x,     person_rate, w, label="Person",        color="#888888", edgecolor="black", hatch="///")
    ax.bar(x + w, other_rate,  w, label="Other Object",  color="#444444", edgecolor="black", hatch="...")
    ax.set_xticks(x); ax.set_xticklabels(["Left", "Middle", "Right"])
    ax.set_ylabel("Fraction of Supervised Frames")
    ax.set_title(title, fontsize=14, pad=10)
    ax.legend(framealpha=0.9)
    savefig(filename)


plot_person_vs_other_rate(held_out_counts,
    "Class rate by camera direction\n(held-out test set)",
    "dataset_class_rate_heldout")


  saved -> /kaggle/working/paper_figures/dataset_class_rate_heldout.pdf


## Section 6 — Model architecture definitions (needed for loading weights)

In [39]:
ACT_MAP = {"relu": nn.ReLU, "gelu": nn.GELU, "leakyrelu": lambda: nn.LeakyReLU(0.1)}


class FlexBackbone(nn.Module):
    BASE_CHANNELS = [16, 32, 64, 128, 256]
    def __init__(self, depth=3, width_mult=1.0, act="relu"):
        super().__init__()
        act_fn = ACT_MAP[act]
        channels = [max(4, int(c * width_mult)) for c in self.BASE_CHANNELS[:depth]]
        self.blocks = nn.ModuleList()
        in_ch = 1
        for i, out_ch in enumerate(channels):
            pool = (i < depth - 1)
            blk = [nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=1),
                                  nn.BatchNorm2d(out_ch), act_fn())]
            if pool: blk.append(nn.MaxPool2d(2))
            self.blocks.append(nn.Sequential(*blk))
            in_ch = out_ch
        self.out_channels = channels

    def forward(self, x):
        feats = []
        for block in self.blocks:
            x = block(x); feats.append(x)
        return feats


class ThermalBackbone(nn.Module):
    def __init__(self, in_channels=2, pool_size=(3,4)):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels,16,5,padding=2); self.bn1=nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16,32,3,padding=1);          self.bn2=nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32,64,3,padding=1);          self.bn3=nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64,64,3,padding=1);          self.bn4=nn.BatchNorm2d(64)
        self.conv5 = nn.Conv2d(64,64,3,padding=1);          self.bn5=nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2); self.pool_size = pool_size
    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.relu(self.bn5(self.conv5(x)))
        x = F.adaptive_avg_pool2d(x, self.pool_size)
        return x.flatten(1)


class QuadrantHead3Class(nn.Module):
    def __init__(self, in_features=64*3*4, num_classes=NUM_CLASSES):
        super().__init__()
        self.fc1=nn.Linear(in_features,64); self.dropout=nn.Dropout(0.3)
        self.fc2=nn.Linear(64, num_classes)
    def forward(self, x):
        return self.fc2(self.dropout(F.relu(self.fc1(x))))


class StitchedBaselineModel(nn.Module):
    def __init__(self, crop_size=(24,32)):
        super().__init__()
        self.backbone = ThermalBackbone(in_channels=2)
        self.head = QuadrantHead3Class()
        self.crop_size = crop_size
    def _extract_quadrant(self, x, bbox):
        x1,y1,x2,y2 = (int(round(v)) for v in bbox)
        x1,y1 = max(x1,0),max(y1,0)
        patch = x[:,:,y1:y2,x1:x2]
        if patch.shape[-2:] != self.crop_size:
            patch = F.interpolate(patch, size=self.crop_size, mode="nearest")
        return patch
    def forward(self, x):
        return torch.stack([self.head(self.backbone(
            self._extract_quadrant(x, CAM_FOOTPRINTS[c]["bbox"])))
            for c in CAM_ORDER_LEFT_TO_RIGHT], dim=1)


class PerCameraHead(nn.Module):
    def __init__(self, feature_dims, hidden=64, dropout=0.3, n_out=NUM_CLASSES):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Linear(sum(feature_dims), hidden), nn.ReLU(),
                                 nn.Dropout(dropout), nn.Linear(hidden, n_out))
    def forward(self, features):
        return self.fc(torch.cat([self.pool(f).flatten(1) for f in features], dim=1))


class OutputFusionModel(nn.Module):
    def __init__(self, depth=3, width_mult=1.0, act="relu",
                 head_hidden=64, head_dropout=0.3, fusion_hidden=64):
        super().__init__()
        self.backbone = FlexBackbone(depth, width_mult, act)
        per_dims = self.backbone.out_channels
        self.per_cam_heads = nn.ModuleList([
            PerCameraHead(per_dims, hidden=head_hidden, dropout=head_dropout)
            for _ in range(NUM_CAMS)])
        fusion_in = NUM_CAMS * NUM_CLASSES
        self.fusion_mlp = nn.Sequential(
            nn.Linear(fusion_in, fusion_hidden), nn.ReLU(), nn.Dropout(head_dropout),
            nn.Linear(fusion_hidden, fusion_in))
    def forward(self, x):
        per_cam = [self.per_cam_heads[i](self.backbone(x[:,i])) for i in range(NUM_CAMS)]
        return self.fusion_mlp(torch.cat(per_cam, dim=1)).view(-1, NUM_CAMS, NUM_CLASSES)


class FlexClassHead3Class(nn.Module):
    def __init__(self, feature_dims, hidden=128, dropout=0.3, n_out_per_cam=NUM_CLASSES):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.n_out_per_cam = n_out_per_cam
        self.fc = nn.Sequential(nn.Linear(sum(feature_dims), hidden), nn.ReLU(),
                                 nn.Dropout(dropout), nn.Linear(hidden, NUM_CAMS*n_out_per_cam))
    def forward(self, features):
        return self.fc(torch.cat([self.pool(f).flatten(1) for f in features],
                                  dim=1)).view(-1, NUM_CAMS, self.n_out_per_cam)


class FeatureFusionModel(nn.Module):
    def __init__(self, depth=3, width_mult=1.0, act="relu", head_hidden=128, head_dropout=0.3):
        super().__init__()
        self.backbone = FlexBackbone(depth, width_mult, act)
        per_dims = self.backbone.out_channels
        self.proj = nn.ModuleList([nn.Conv2d(d*NUM_CAMS, d, 1) for d in per_dims])
        self.head = FlexClassHead3Class(per_dims, hidden=head_hidden, dropout=head_dropout)
    def forward(self, x):
        n = len(self.backbone.out_channels)
        all_feats = [[] for _ in range(n)]
        for i in range(NUM_CAMS):
            for lvl, f in enumerate(self.backbone(x[:,i])):
                all_feats[lvl].append(f)
        fused = [proj(torch.cat(lf, dim=1)) for proj, lf in zip(self.proj, all_feats)]
        return self.head(fused)


class CameraIdentityClassifier(nn.Module):
    def __init__(self, num_directions=3):
        super().__init__()
        self.conv1=nn.Conv2d(1,16,3,padding=1); self.bn1=nn.BatchNorm2d(16)
        self.conv2=nn.Conv2d(16,32,3,padding=1); self.bn2=nn.BatchNorm2d(32)
        self.conv3=nn.Conv2d(32,64,3,padding=1); self.bn3=nn.BatchNorm2d(64)
        self.pool=nn.MaxPool2d(2)
        self.fc1=nn.Linear(64,32); self.dropout=nn.Dropout(0.3); self.fc2=nn.Linear(32,num_directions)
    def forward(self, x):
        x=self.pool(F.relu(self.bn1(self.conv1(x))))
        x=self.pool(F.relu(self.bn2(self.conv2(x))))
        x=F.relu(self.bn3(self.conv3(x)))
        x=F.adaptive_avg_pool2d(x,1).flatten(1)
        return self.fc2(self.dropout(F.relu(self.fc1(x))))


print("All model classes defined.")


All model classes defined.


## Section 7 — Load trained models

In [40]:
def load_model(model, filename):
    path = MODELS_DIR / filename
    model.load_state_dict(torch.load(path, map_location=device))
    model.to(device).eval()
    print(f"  Loaded {filename}  ({sum(p.numel() for p in model.parameters()):,} params)")
    return model


best_combo_config = pickle.load(open(RESULTS_DIR / "best_combo_config.pkl", "rb"))
print("Best combo config:", best_combo_config)

baseline1_model = load_model(StitchedBaselineModel(), "baseline1_stitched.pt")
baseline2_model = load_model(OutputFusionModel(), "baseline2_output_fusion.pt")
ff_model        = load_model(FeatureFusionModel(**best_combo_config), "ff_best_combo.pt")
identity_model  = load_model(CameraIdentityClassifier(), "camera_identity_classifier.pt")


Best combo config: {'depth': 5, 'width_mult': 1.5, 'act': 'gelu', 'head_hidden': 512, 'head_dropout': 0.1}
  Loaded baseline1_stitched.pt  (147,699 params)
  Loaded baseline2_output_fusion.pt  (47,026 params)
  Loaded ff_best_combo.pt  (1,859,777 params)
  Loaded camera_identity_classifier.pt  (25,699 params)


## Section 8 — Held-out datasets & loaders

In [41]:
_RAW_SEQUENCES = {}

def _get_raw_sequences(part):
    if part not in _RAW_SEQUENCES:
        thermal_dir = HELD_OUT_PARTS[part]["thermal_dir"]
        _RAW_SEQUENCES[part] = {
            c: [grid for _,_,grid in build_camera_sequence(thermal_dir, c)]
            for c in ACTIVE_CAMERAS}
    return _RAW_SEQUENCES[part]


def load_tiles(part, anchor_index):
    seqs = _get_raw_sequences(part)
    return [seqs[c][anchor_index].astype(np.float32) for c in CAM_ORDER_LEFT_TO_RIGHT]


def build_stitched_tensor(tiles):
    # stitch for Baseline 1
    from functools import reduce
    import cv2 as _cv2
    frames = tiles
    shapes = [(SENSOR_H, SENSOR_W)] * NUM_CAMS
    canvas = np.zeros((PANO_H, PANO_W), dtype=np.float32)
    for tile, T in zip(frames, main_cluster["transforms"]):
        full_T = PANO_TRANSLATION @ T
        warped = _cv2.warpPerspective(tile.astype(np.float64), full_T, (PANO_W, PANO_H),
                                       flags=_cv2.INTER_NEAREST, borderMode=_cv2.BORDER_CONSTANT)
        mask = warped > 0
        canvas[mask] = warped[mask].astype(np.float32)
    mask_ch = (canvas > 0).astype(np.float32)
    return np.stack([canvas, mask_ch], axis=0)


class HeldOutTileDataset(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        tiles = load_tiles(row["_part"], int(row["anchor_index"]))
        tensor = np.stack([t[None] for t in tiles], axis=0)
        class_vec, mask_vec = get_held_out_label(row)
        return torch.from_numpy(tensor), torch.from_numpy(class_vec), torch.from_numpy(mask_vec)


class HeldOutStitchedDataset(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        tiles = load_tiles(row["_part"], int(row["anchor_index"]))
        tensor = build_stitched_tensor(tiles)
        class_vec, mask_vec = get_held_out_label(row)
        return torch.from_numpy(tensor), torch.from_numpy(class_vec), torch.from_numpy(mask_vec)


tile_ds     = HeldOutTileDataset(held_out_rows)
stitched_ds = HeldOutStitchedDataset(held_out_rows)

tile_ld     = DataLoader(tile_ds,     batch_size=64, shuffle=False, num_workers=2)
stitched_ld = DataLoader(stitched_ds, batch_size=64, shuffle=False, num_workers=2)
print(f"Held-out loaders ready: {len(tile_ld)} tile batches, {len(stitched_ld)} stitched batches")


Held-out loaders ready: 16 tile batches, 16 stitched batches


## Section 9 — Evaluation helpers

In [42]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_logits, all_labels, all_masks = [], [], []
    for x, y, m in loader:
        logits = model(x.to(device)).cpu()
        all_logits.append(logits); all_labels.append(y); all_masks.append(m)
    logits = torch.cat(all_logits); labels = torch.cat(all_labels); masks = torch.cat(all_masks)
    preds  = logits.argmax(dim=-1)
    p_np, t_np, m_np = preds.numpy(), labels.numpy(), masks.numpy()
    return {"preds": p_np, "trues": t_np, "masks": m_np}


def masked_macro_f1(preds, trues, masks, classes=(1,2)):
    cam_f1s = []
    for i in range(preds.shape[1]):
        valid = masks[:,i] > 0
        if not valid.any(): continue
        pc = f1_score(trues[valid,i], preds[valid,i], labels=list(classes),
                       average=None, zero_division=0)
        cam_f1s.append(float(np.mean(pc)))
    return float(np.mean(cam_f1s)) if cam_f1s else 0.0


def per_class_f1_table(result, model_name, split_label):
    '''Returns a dataframe with per-direction, per-class F1 scores.'''
    p, t, m = result["preds"], result["trues"], result["masks"]
    rows = []
    for i, cam_id in enumerate(CAM_ORDER_LEFT_TO_RIGHT):
        valid = m[:,i] > 0
        if not valid.any():
            continue
        per_class = f1_score(t[valid,i], p[valid,i], labels=[0,1,2],
                              average=None, zero_division=0)
        rows.append({
            "Model": model_name,
            "Split": split_label,
            "Direction": CAM_TO_DIRECTION[cam_id].capitalize(),
            "F1 (no object)": round(per_class[0], 3),
            "F1 (person)": round(per_class[1], 3),
            "F1 (other object)": round(per_class[2], 3),
            "Macro F1 (person+other)": round(float(np.mean([per_class[1], per_class[2]])), 3),
        })
    # add overall row
    overall_per_class = [
        f1_score(t[m[:,i]>0, i], p[m[:,i]>0, i], labels=[c], average="binary", zero_division=0)
        for c in [0,1,2]
        for i in range(NUM_CAMS) if (m[:,i]>0).any()
    ]
    macro = masked_macro_f1(p, t, m)
    rows.append({
        "Model": model_name, "Split": split_label, "Direction": "Overall",
        "F1 (no object)": "—", "F1 (person)": "—", "F1 (other object)": "—",
        "Macro F1 (person+other)": round(macro, 3),
    })
    return pd.DataFrame(rows)


def measure_latency(model, dummy, n_runs=200):
    m = model.cpu().eval()
    with torch.no_grad():
        for _ in range(20): m(dummy)
        t0 = time.perf_counter()
        for _ in range(n_runs): m(dummy)
    model.to(device)
    return (time.perf_counter() - t0) / n_runs * 1000


print("Evaluation helpers defined.")


Evaluation helpers defined.


## Section 10 — Evaluate all 4 models on held-out test set

In [43]:
print("Evaluating Baseline 1 (Stitched)...")
b1_test = evaluate(baseline1_model, stitched_ld)
b1_test["macro_f1"] = masked_macro_f1(b1_test["preds"], b1_test["trues"], b1_test["masks"])

print("Evaluating Baseline 2 (Output Fusion)...")
b2_test = evaluate(baseline2_model, tile_ld)
b2_test["macro_f1"] = masked_macro_f1(b2_test["preds"], b2_test["trues"], b2_test["masks"])

print("Evaluating Feature Fusion (best combo)...")
ff_test = evaluate(ff_model, tile_ld)
ff_test["macro_f1"] = masked_macro_f1(ff_test["preds"], ff_test["trues"], ff_test["masks"])

print(f"\nHeld-out macro F1:")
print(f"  Baseline 1:      {b1_test['macro_f1']:.4f}")
print(f"  Baseline 2:      {b2_test['macro_f1']:.4f}")
print(f"  Feature Fusion:  {ff_test['macro_f1']:.4f}")


Evaluating Baseline 1 (Stitched)...
Evaluating Baseline 2 (Output Fusion)...
Evaluating Feature Fusion (best combo)...

Held-out macro F1:
  Baseline 1:      0.4170
  Baseline 2:      0.4363
  Feature Fusion:  0.9110


## Section 11 — Shuffle robustness on held-out set (blind / classifier / oracle)

In [44]:
class ShuffleDataset(Dataset):
    def __init__(self, base_ds, fixed_permutation=None):
        self.base = base_ds
        self.canonical = list(CAM_ORDER_LEFT_TO_RIGHT)
        self.perm = fixed_permutation
    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        tensor, label_vec, mask_vec = self.base[idx]
        perm = [self.canonical.index(c) for c in self.perm] if self.perm else list(range(NUM_CAMS))
        slot_cam_ids = torch.tensor([self.canonical[i] for i in perm], dtype=torch.long)
        return tensor[perm], label_vec[list(range(NUM_CAMS))], mask_vec[list(range(NUM_CAMS))], slot_cam_ids


@torch.no_grad()
def classifier_resort(identity_model, tensor):
    B = tensor.shape[0]
    flat = tensor.reshape(B*NUM_CAMS, *tensor.shape[2:]).to(device)
    log_probs = F.log_softmax(identity_model(flat), dim=1).cpu().view(B, NUM_CAMS, NUM_CAMS)
    all_perms = list(permutations(range(NUM_CAMS)))
    resorted = torch.empty_like(tensor)
    for b in range(B):
        best_score, best_perm = -float("inf"), all_perms[0]
        for perm in all_perms:
            score = sum(log_probs[b, perm[d], d].item() for d in range(NUM_CAMS))
            if score > best_score:
                best_score, best_perm = score, perm
        resorted[b] = tensor[b, list(best_perm)]
    return resorted


@torch.no_grad()
def eval_shuffle(model, base_ds, perm, mode, identity_model=None):
    '''Returns macro F1 for a specific permutation and correction mode.'''
    ds = ShuffleDataset(base_ds, fixed_permutation=perm)
    ld = DataLoader(ds, batch_size=64, shuffle=False, num_workers=0)
    all_preds, all_trues, all_masks = [], [], []
    canonical = list(CAM_ORDER_LEFT_TO_RIGHT)
    for tensor, label_vec, mask_vec, slot_cam_ids in ld:
        if mode == "oracle":
            resort_idx = torch.stack([
                torch.tensor([slot_cam_ids[b].tolist().index(c) for c in canonical])
                for b in range(tensor.shape[0])])
            tensor = torch.stack([tensor[b, resort_idx[b]] for b in range(tensor.shape[0])])
        elif mode == "classifier":
            tensor = classifier_resort(identity_model, tensor)
        logits = model(tensor.to(device)).cpu()
        all_preds.append(logits.argmax(dim=-1))
        all_trues.append(label_vec)
        all_masks.append(mask_vec)
    p = torch.cat(all_preds).numpy()
    t = torch.cat(all_trues).numpy()
    m = torch.cat(all_masks).numpy()
    return masked_macro_f1(p, t, m)


all_perms = list(permutations(CAM_ORDER_LEFT_TO_RIGHT))
print(f"Running shuffle evaluation across {len(all_perms)} permutations × 3 modes...")

shuffle_results = {}
for perm in all_perms:
    for mode in ["blind", "classifier", "oracle"]:
        key = (perm, mode)
        kwargs = {"identity_model": identity_model} if mode == "classifier" else {}
        shuffle_results[key] = eval_shuffle(ff_model, tile_ds, perm, mode, **kwargs)
        print(f"  perm={perm}  mode={mode}  macro_f1={shuffle_results[key]:.4f}")

canonical_perm = tuple(CAM_ORDER_LEFT_TO_RIGHT)
mean_blind      = np.mean([shuffle_results[(p,"blind")] for p in all_perms])
mean_classifier = np.mean([shuffle_results[(p,"classifier")] for p in all_perms])
mean_oracle     = np.mean([shuffle_results[(p,"oracle")] for p in all_perms])
print(f"\nMean across all permutations:")
print(f"  Blind:       {mean_blind:.4f}")
print(f"  Classifier:  {mean_classifier:.4f}")
print(f"  Oracle:      {mean_oracle:.4f}")


Running shuffle evaluation across 6 permutations × 3 modes...
  perm=(2, 5, 4)  mode=blind  macro_f1=0.9110
  perm=(2, 5, 4)  mode=classifier  macro_f1=0.9092
  perm=(2, 5, 4)  mode=oracle  macro_f1=0.9110
  perm=(2, 4, 5)  mode=blind  macro_f1=0.3206
  perm=(2, 4, 5)  mode=classifier  macro_f1=0.9092
  perm=(2, 4, 5)  mode=oracle  macro_f1=0.9110
  perm=(5, 2, 4)  mode=blind  macro_f1=0.2707
  perm=(5, 2, 4)  mode=classifier  macro_f1=0.9092
  perm=(5, 2, 4)  mode=oracle  macro_f1=0.9110
  perm=(5, 4, 2)  mode=blind  macro_f1=0.2047
  perm=(5, 4, 2)  mode=classifier  macro_f1=0.9092
  perm=(5, 4, 2)  mode=oracle  macro_f1=0.9110
  perm=(4, 2, 5)  mode=blind  macro_f1=0.2014
  perm=(4, 2, 5)  mode=classifier  macro_f1=0.9092
  perm=(4, 2, 5)  mode=oracle  macro_f1=0.9110
  perm=(4, 5, 2)  mode=blind  macro_f1=0.5607
  perm=(4, 5, 2)  mode=classifier  macro_f1=0.9092
  perm=(4, 5, 2)  mode=oracle  macro_f1=0.9110

Mean across all permutations:
  Blind:       0.4115
  Classifier:  0.9092

## Section 12 — Paper tables

Two tables: one for validation set (from cached DeepNet results), one for held-out test set.
Both show per-class F1 (person / other object) separately, plus macro.

In [45]:
from matplotlib.patches import Patch

val_results = {
    "baseline1": pickle.load(open(RESULTS_DIR/"baseline1_stitched_result.pkl","rb")),
    "baseline2": pickle.load(open(RESULTS_DIR/"baseline2_output_fusion_result.pkl","rb")),
    "ff_best":   pickle.load(open(RESULTS_DIR/"ff_best_combo_result.pkl","rb")),
}

COLOR_SHADES  = ["#4477AA", "#66CCEE", "#228833", "#CCBB44", "#EE6677", "#AA3377"]
COLOR_HATCHES = ["",        "///",     "...",      "xxx",     "---",     "|||"]
GRAY_SHADES   = ["#FFFFFF", "#CCCCCC", "#888888",  "#FFFFFF", "#AAAAAA", "#333333"]
GRAY_HATCHES  = ["",        "///",     "...",      "xxx",     "---",     "|||"]
CLASS_LABELS  = ["Clear", "Person", "Other Object"]
CLASS_KEYS    = [0, 1, 2]

def get_per_dir_f1_by_class(result):
    p, t, m = result["preds"], result["trues"], result["masks"]
    out = {}
    class_accum = {0: [], 1: [], 2: []}
    for i, cam_id in enumerate(CAM_ORDER_LEFT_TO_RIGHT):
        valid = m[:, i] > 0
        if not valid.any():
            out[CAM_TO_DIRECTION[cam_id]] = {0: 0.0, 1: 0.0, 2: 0.0}
            continue
        pc = f1_score(t[valid, i], p[valid, i], labels=[0, 1, 2], average=None, zero_division=0)
        out[CAM_TO_DIRECTION[cam_id]] = {c: float(pc[c]) for c in range(3)}
        for c in range(3):
            class_accum[c].append(float(pc[c]))
    out["overall"] = {c: float(np.mean(class_accum[c])) if class_accum[c] else 0.0 for c in range(3)}
    return out

@torch.no_grad()
def shuffle_per_dir_by_class(model, base_ds, mode, identity_model=None):
    dir_vals = {d: {0: [], 1: [], 2: []} for d in DIRECTIONS + ["overall"]}
    for perm in permutations(CAM_ORDER_LEFT_TO_RIGHT):
        ds = ShuffleDataset(base_ds, fixed_permutation=list(perm))
        ld = DataLoader(ds, batch_size=64, shuffle=False, num_workers=0)
        all_p, all_t, all_m = [], [], []
        canonical = list(CAM_ORDER_LEFT_TO_RIGHT)
        for tensor, label_vec, mask_vec, slot_cam_ids in ld:
            if mode == "oracle":
                ri = torch.stack([torch.tensor(
                    [slot_cam_ids[b].tolist().index(c) for c in canonical])
                    for b in range(tensor.shape[0])])
                tensor = torch.stack([tensor[b, ri[b]] for b in range(tensor.shape[0])])
            elif mode == "classifier":
                tensor = classifier_resort(identity_model, tensor)
            logits = model(tensor.to(device)).cpu()
            all_p.append(logits.argmax(dim=-1))
            all_t.append(label_vec); all_m.append(mask_vec)
        p_np = torch.cat(all_p).numpy()
        t_np = torch.cat(all_t).numpy()
        m_np = torch.cat(all_m).numpy()
        cam_class = {0: [], 1: [], 2: []}
        for i, cam_id in enumerate(CAM_ORDER_LEFT_TO_RIGHT):
            valid = m_np[:, i] > 0
            if not valid.any():
                continue
            pc = f1_score(t_np[valid, i], p_np[valid, i], labels=[0, 1, 2], average=None, zero_division=0)
            d = CAM_TO_DIRECTION[cam_id]
            for c in range(3):
                dir_vals[d][c].append(float(pc[c]))
                cam_class[c].append(float(pc[c]))
        for c in range(3):
            dir_vals["overall"][c].append(float(np.mean(cam_class[c])) if cam_class[c] else 0.0)
    return {d: {c: float(np.mean(dir_vals[d][c])) if dir_vals[d][c] else 0.0 for c in range(3)}
            for d in DIRECTIONS + ["overall"]}

print("Computing per-direction shuffle F1s (all 3 classes)...")
shuf_blind = shuffle_per_dir_by_class(ff_model, tile_ds, "blind")
shuf_clf   = shuffle_per_dir_by_class(ff_model, tile_ds, "classifier", identity_model)
shuf_ora   = shuffle_per_dir_by_class(ff_model, tile_ds, "oracle")
print("Done.")

models_data_by_class = {
    "Baseline 1\n(Stitched)":        get_per_dir_f1_by_class(val_results["baseline1"]),
    "Baseline 2\n(Output Fusion)":   get_per_dir_f1_by_class(val_results["baseline2"]),
    "Feature Fusion\n(Best Combo)":  get_per_dir_f1_by_class(val_results["ff_best"]),
    "FF + Shuffle\n(Blind)":         shuf_blind,
    "FF + Shuffle\n(Classifier)":    shuf_clf,
    "FF + Shuffle\n(Oracle)":        shuf_ora,
}
model_names = list(models_data_by_class.keys())
n_models = len(model_names)
groups = DIRECTIONS + ["overall"]
group_labels = ["Left", "Middle", "Right", "Overall"]


CLASS_PANEL_LABELS = ["Clear", "Person", "Other Object"]

FIG_H = 2.75
BAR_KW = dict(edgecolor="black", linewidth=1.0)
LABEL_FMT = "{:.2f}"
LABEL_FONTSIZE = 6
LABEL_PAD = 0.018
LABEL_COLLISION_DY = 0.028
YLIM_HEADROOM = 0.03

FIG1_CAPTION = (
    "Per-class F1 by camera direction for six obstruction-detection architectures. "
    "Baselines 1–2 and Feature Fusion are evaluated on the validation set; "
    "Feature Fusion with shuffle correction (blind, classifier, oracle) is evaluated "
    "on the held-out test set."
)
FIG2_CAPTION = (
    "Per-class F1 of Feature Fusion with classifier-based shuffle correction on the "
    "held-out test set, broken down by camera direction (left, middle, right) and overall."
)


def _style_axes(ax, ylabel=None):
    ax.set_yticks([0, 0.5, 1.0])
    ax.yaxis.grid(True, linestyle="--", color="#AAAAAA", alpha=0.5, linewidth=0.8)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)
        spine.set_color("black")
    if ylabel:
        ax.set_ylabel(ylabel)


def _trim_xaxis(ax, n_groups):
    ax.set_xlim(-0.52, n_groups - 0.48)
    ax.margins(x=0)


def _group_separators(ax, n_groups):
    for xi in range(1, n_groups):
        ax.axvline(xi - 0.5, color="#AAAAAA", linestyle=":", linewidth=0.8, alpha=0.7)


def _add_bar_top_labels(ax, bar_groups, val_groups):
    bar_w = bar_groups[0][0].get_width()
    max_label_y = 0.0
    for gi in range(len(val_groups[0])):
        entries = []
        for bars, vals in zip(bar_groups, val_groups):
            bar = bars[gi]
            val = float(vals[gi])
            cx = bar.get_x() + bar.get_width() / 2
            entries.append([cx, val, val + LABEL_PAD])
        entries.sort(key=lambda e: e[0])
        for i in range(1, len(entries)):
            cx_prev, _, y_prev = entries[i - 1]
            cx, val, y = entries[i]
            if abs(cx - cx_prev) < bar_w * 1.05 and abs(y - y_prev) < LABEL_COLLISION_DY:
                entries[i][2] = y_prev + LABEL_COLLISION_DY
        for cx, val, y in entries:
            max_label_y = max(max_label_y, y)
            ax.text(cx, y, LABEL_FMT.format(val), fontsize=LABEL_FONTSIZE,
                    fontweight="bold", ha="center", va="bottom", clip_on=False)
    return max_label_y


def _set_ylim_with_labels(ax, max_label_y):
    ax.set_ylim(0, max(1.0, max_label_y + YLIM_HEADROOM))


def _legend_below(fig, handles, labels, ncol=3):
    fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.01),
               ncol=ncol, frameon=True, edgecolor="black", facecolor="white",
               columnspacing=0.9, handlelength=1.2, handletextpad=0.4)


def _finalize_figure(fig, bottom=0.28):
    fig.subplots_adjust(left=0.07, right=0.99, top=0.94, bottom=bottom, wspace=0.18)


def draw_fig1(shades, hatches, suffix):
    fig, axes = plt.subplots(1, 3, figsize=(9.8, FIG_H), sharey=True)
    x = np.arange(len(groups))
    w = 0.125
    offsets = np.linspace(-(n_models-1)/2*w, (n_models-1)/2*w, n_models)
    global_max_label_y = 0.0
    for ax, cls_key, panel_label in zip(axes, CLASS_KEYS, CLASS_PANEL_LABELS):
        bar_groups, val_groups = [], []
        for name, shade, hatch, offset in zip(model_names, shades, hatches, offsets):
            vals = [models_data_by_class[name][g][cls_key] for g in groups]
            bars = ax.bar(x + offset, vals, w, color=shade, hatch=hatch, label=name, **BAR_KW)
            bar_groups.append(bars)
            val_groups.append(vals)
        global_max_label_y = max(global_max_label_y, _add_bar_top_labels(ax, bar_groups, val_groups))
        _group_separators(ax, len(groups))
        _trim_xaxis(ax, len(groups))
        ax.set_xticks(x)
        ax.set_xticklabels(group_labels)
        _style_axes(ax, panel_label)
        if ax is not axes[0]:
            ax.tick_params(axis="y", labelleft=False)
    for ax in axes:
        _set_ylim_with_labels(ax, global_max_label_y)
    handles, labels_leg = axes[0].get_legend_handles_labels()
    _legend_below(fig, handles, labels_leg, ncol=3)
    _finalize_figure(fig, bottom=0.28)
    for ext in ["pdf", "png"]:
        kw = {"dpi": 200} if ext == "png" else {}
        plt.savefig(OUT_DIR / f"fig1_macro_f1_{suffix}.{ext}",
                    bbox_inches="tight", pad_inches=0.04, facecolor="white", **kw)
    plt.show()
    print(f"  saved -> fig1_macro_f1_{suffix}")

draw_fig1(COLOR_SHADES, COLOR_HATCHES, "color")
draw_fig1(GRAY_SHADES,  GRAY_HATCHES,  "grayscale")


Computing per-direction shuffle F1s...
Done.
  saved -> fig1_macro_f1_color
  saved -> fig1_macro_f1_grayscale


In [46]:
# ---- load cached val results from DeepNet run ----
val_results = {
    "baseline1": pickle.load(open(RESULTS_DIR/"baseline1_stitched_result.pkl","rb")),
    "baseline2": pickle.load(open(RESULTS_DIR/"baseline2_output_fusion_result.pkl","rb")),
    "ff_best":   pickle.load(open(RESULTS_DIR/"ff_best_combo_result.pkl","rb")),
}

MODEL_DISPLAY_NAMES = {
    "baseline1": "Baseline 1 (Stitched)",
    "baseline2": "Baseline 2 (Output Fusion)",
    "ff_best":   "Feature Fusion (best combo)",
}

# ---- VAL TABLE ----
val_table_rows = []
for key, name in MODEL_DISPLAY_NAMES.items():
    r = val_results[key]
    p, t, m = r["preds"], r["trues"], r["masks"]
    for i, cam_id in enumerate(CAM_ORDER_LEFT_TO_RIGHT):
        valid = m[:,i] > 0
        if not valid.any(): continue
        pc = f1_score(t[valid,i], p[valid,i], labels=[0,1,2], average=None, zero_division=0)
        val_table_rows.append({
            "Model": name,
            "Direction": CAM_TO_DIRECTION[cam_id].capitalize(),
            "F1 (no object)":         round(pc[0], 3),
            "F1 (person)":            round(pc[1], 3),
            "F1 (other object)":      round(pc[2], 3),
            "Macro F1\n(person+other)": round(float(np.mean([pc[1],pc[2]])), 3),
        })
    # overall
    val_table_rows.append({
        "Model": name, "Direction": "Overall",
        "F1 (no object)": "—", "F1 (person)": "—", "F1 (other object)": "—",
        "Macro F1\n(person+other)": round(masked_macro_f1(p, t, m), 3),
    })

val_df = pd.DataFrame(val_table_rows)
print("=== VALIDATION SET RESULTS ===")
print(val_df.to_string(index=False))

# ---- TEST TABLE ----
test_table_rows = []
for result, name in [(b1_test,"Baseline 1 (Stitched)"),
                      (b2_test,"Baseline 2 (Output Fusion)"),
                      (ff_test,"Feature Fusion (best combo)")]:
    p, t, m = result["preds"], result["trues"], result["masks"]
    for i, cam_id in enumerate(CAM_ORDER_LEFT_TO_RIGHT):
        valid = m[:,i] > 0
        if not valid.any(): continue
        pc = f1_score(t[valid,i], p[valid,i], labels=[0,1,2], average=None, zero_division=0)
        test_table_rows.append({
            "Model": name,
            "Direction": CAM_TO_DIRECTION[cam_id].capitalize(),
            "F1 (no object)":         round(pc[0], 3),
            "F1 (person)":            round(pc[1], 3),
            "F1 (other object)":      round(pc[2], 3),
            "Macro F1\n(person+other)": round(float(np.mean([pc[1],pc[2]])), 3),
        })
    test_table_rows.append({
        "Model": name, "Direction": "Overall",
        "F1 (no object)": "—", "F1 (person)": "—", "F1 (other object)": "—",
        "Macro F1\n(person+other)": round(masked_macro_f1(p, t, m), 3),
    })

test_df = pd.DataFrame(test_table_rows)
print("\n=== HELD-OUT TEST SET RESULTS ===")
print(test_df.to_string(index=False))

# ---- SHUFFLE TABLE (test set only, as requested) ----
shuffle_table = []
for perm in all_perms:
    perm_label = f"({', '.join(CAM_TO_DIRECTION[c] for c in perm)})"
    for mode in ["blind","classifier","oracle"]:
        shuffle_table.append({
            "Permutation": perm_label,
            "Correction": mode.capitalize(),
            "Macro F1 (person+other)": round(shuffle_results[(perm, mode)], 3),
        })
shuffle_df = pd.DataFrame(shuffle_table)
print("\n=== SHUFFLE ROBUSTNESS (held-out test set) ===")
print(shuffle_df.to_string(index=False))

# Summary shuffle table
print("\n=== SHUFFLE SUMMARY (mean across all permutations) ===")
summary_shuffle = pd.DataFrame([
    {"Correction": "Blind (no correction)",      "Mean Macro F1": round(mean_blind,       4)},
    {"Correction": "Classifier-corrected",        "Mean Macro F1": round(mean_classifier,  4)},
    {"Correction": "Oracle-corrected (upper bound)", "Mean Macro F1": round(mean_oracle,   4)},
])
print(summary_shuffle.to_string(index=False))

# save tables as CSV for easy LaTeX import
val_df.to_csv(OUT_DIR / "table_val_results.csv", index=False)
test_df.to_csv(OUT_DIR / "table_test_results.csv", index=False)
summary_shuffle.to_csv(OUT_DIR / "table_shuffle_summary.csv", index=False)
print("\nTables saved to CSV.")


=== VALIDATION SET RESULTS ===
                      Model Direction F1 (no object) F1 (person) F1 (other object)  Macro F1\n(person+other)
      Baseline 1 (Stitched)      Left          0.786       0.785             0.739                     0.762
      Baseline 1 (Stitched)    Middle          0.706       0.842             0.769                     0.805
      Baseline 1 (Stitched)     Right          0.851       0.591             0.692                     0.641
      Baseline 1 (Stitched)   Overall              —           —                 —                     0.736
 Baseline 2 (Output Fusion)      Left          0.821       0.872             0.768                     0.820
 Baseline 2 (Output Fusion)    Middle           0.56       0.822             0.767                     0.794
 Baseline 2 (Output Fusion)     Right          0.847       0.612             0.678                     0.645
 Baseline 2 (Output Fusion)   Overall              —           —                 —               

## Section 13 — Paper figures

Exactly two bar charts (no plot titles; use captions in the paper):

1. **Architecture comparison** — three side-by-side panels (Clear, Person, Other Object F1) for Baseline 1, Baseline 2, Feature Fusion (best combo) on **val**, plus FF+Shuffle blind/classifier/oracle on **held-out test**.
2. **Best model per-class breakdown** — Feature Fusion + shuffle classifier on held-out test, all three classes by camera direction.


In [ ]:
DIRECTION_TO_CAM = {v: k for k, v in CAM_TO_DIRECTION.items()}

class ShuffleDatasetSingle(torch.utils.data.Dataset):
    def __init__(self, base_ds):
        self.base = base_ds
    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        tensor, label_vec, mask_vec = self.base[idx]
        slot_cam_ids = torch.tensor(list(CAM_ORDER_LEFT_TO_RIGHT), dtype=torch.long)
        return tensor, label_vec, mask_vec, slot_cam_ids

@torch.no_grad()
def eval_canonical_with_classifier(model, identity_model, base_ds):
    ds = ShuffleDatasetSingle(base_ds)
    ld = torch.utils.data.DataLoader(ds, batch_size=64, shuffle=False, num_workers=0)
    all_preds, all_trues, all_masks = [], [], []
    for tensor, label_vec, mask_vec, slot_cam_ids in ld:
        tensor_resorted = classifier_resort(identity_model, tensor)
        logits = model(tensor_resorted.to(device)).cpu()
        all_preds.append(logits.argmax(dim=-1))
        all_trues.append(label_vec); all_masks.append(mask_vec)
    return (torch.cat(all_preds).numpy(),
            torch.cat(all_trues).numpy(),
            torch.cat(all_masks).numpy())

p_clf, t_clf, m_clf = eval_canonical_with_classifier(ff_model, identity_model, tile_ds)

groups = DIRECTIONS + ["overall"]
group_labels = ["Left", "Middle", "Right", "Overall"]

noobj_f1, person_f1, other_f1 = [], [], []
for g in groups:
    if g == "overall":
        per_class = [[], [], []]
        for i in range(NUM_CAMS):
            valid = m_clf[:, i] > 0
            if not valid.any(): continue
            pc = f1_score(t_clf[valid, i], p_clf[valid, i], labels=[0,1,2],
                          average=None, zero_division=0)
            for c in range(3): per_class[c].append(float(pc[c]))
        noobj_f1.append(float(np.mean(per_class[0])) if per_class[0] else 0.0)
        person_f1.append(float(np.mean(per_class[1])) if per_class[1] else 0.0)
        other_f1.append(float(np.mean(per_class[2])) if per_class[2] else 0.0)
    else:
        cam_id = DIRECTION_TO_CAM[g]
        i = CAM_ORDER_LEFT_TO_RIGHT.index(cam_id)
        valid = m_clf[:, i] > 0
        if not valid.any():
            noobj_f1.append(0.0); person_f1.append(0.0); other_f1.append(0.0); continue
        pc = f1_score(t_clf[valid, i], p_clf[valid, i], labels=[0,1,2],
                       average=None, zero_division=0)
        noobj_f1.append(float(pc[0]))
        person_f1.append(float(pc[1]))
        other_f1.append(float(pc[2]))

x = np.arange(len(groups))
w = 0.28



# ---- export figure metrics (inlined; no external module on Kaggle) ----
_CLASS_JSON_KEYS = ["clear", "person", "other"]

def _class_dict_to_json(class_dict):
    return {_CLASS_JSON_KEYS[k]: float(class_dict[k]) for k in (0, 1, 2)}

def export_figure_metrics(path, models_data_by_class, noobj_f1, person_f1, other_f1,
                          fig1_caption, fig2_caption, source="figures-creation.ipynb"):
    path = Path(path)
    groups = ["left", "middle", "right", "overall"]
    payload = {
        "source": source,
        "models_data_by_class": {
            model_name: {
                direction: _class_dict_to_json(by_dir[direction])
                for direction in groups
            }
            for model_name, by_dir in models_data_by_class.items()
        },
        "fig2": {
            "clear_f1": [float(v) for v in noobj_f1],
            "person_f1": [float(v) for v in person_f1],
            "other_f1": [float(v) for v in other_f1],
        },
        "captions": {"fig1": fig1_caption, "fig2": fig2_caption},
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
    return path



metrics_path = OUT_DIR / "figure_metrics.json"
export_figure_metrics(
    metrics_path,
    models_data_by_class,
    noobj_f1,
    person_f1,
    other_f1,
    FIG1_CAPTION,
    FIG2_CAPTION,
)
print(f"  exported metrics -> {metrics_path}")

COLOR_CLASS = {"Clear": ("#4477AA", ""), "Person": ("#EE6677", "///"), "Other Object": ("#228833", "...")}
GRAY_CLASS  = {"Clear": ("#FFFFFF", ""), "Person": ("#AAAAAA", "///"), "Other Object": ("#444444", "...")}

def draw_fig2(palette, suffix):
    fig, ax = plt.subplots(figsize=(4.9, FIG_H))
    b1 = ax.bar(x-w, noobj_f1,  w, label="Clear",
                color=palette["Clear"][0],  hatch=palette["Clear"][1],  **BAR_KW)
    b2 = ax.bar(x,   person_f1, w, label="Person",
                color=palette["Person"][0],        hatch=palette["Person"][1],        **BAR_KW)
    b3 = ax.bar(x+w, other_f1,  w, label="Other Object",
                color=palette["Other Object"][0],  hatch=palette["Other Object"][1],  **BAR_KW)
    max_label_y = _add_bar_top_labels(ax, [b1, b2, b3], [noobj_f1, person_f1, other_f1])
    _set_ylim_with_labels(ax, max_label_y)
    _group_separators(ax, len(groups))
    _trim_xaxis(ax, len(groups))
    ax.set_xticks(x); ax.set_xticklabels(group_labels)
    _style_axes(ax, "F1 Score")
    handles, labels_leg = ax.get_legend_handles_labels()
    _legend_below(fig, handles, labels_leg, ncol=3)
    _finalize_figure(fig, bottom=0.22)
    for ext in ["pdf", "png"]:
        kw2 = {"dpi": 200} if ext == "png" else {}
        plt.savefig(OUT_DIR / f"fig2_ff_per_class_{suffix}.{ext}",
                    bbox_inches="tight", pad_inches=0.04, facecolor="white", **kw2)
    plt.show()
    print(f"  saved -> fig2_ff_per_class_{suffix}")

draw_fig2(COLOR_CLASS, "color")
draw_fig2(GRAY_CLASS,  "grayscale")

print('Fig. 1 caption:', FIG1_CAPTION)
print('Fig. 2 caption:', FIG2_CAPTION)


In [48]:
# ---- Latency note (text only, no figure per notes) ----
dummy_tile     = torch.zeros(1, NUM_CAMS, 1, SENSOR_H, SENSOR_W)
dummy_stitched = torch.zeros(1, 2, PANO_H, PANO_W)

lat_b1 = measure_latency(baseline1_model, dummy_stitched)
lat_b2 = measure_latency(baseline2_model, dummy_tile)
lat_ff = measure_latency(ff_model, dummy_tile)
lat_id = measure_latency(identity_model, torch.zeros(1, 1, SENSOR_H, SENSOR_W))

print("=" * 55)
print("LATENCY (for paper — text note, no figure)")
print("=" * 55)
print(f"  Baseline 1 (Stitched):       {lat_b1:.2f} ms")
print(f"  Baseline 2 (Output Fusion):  {lat_b2:.2f} ms")
print(f"  Feature Fusion (best combo): {lat_ff:.2f} ms")
print(f"  + Identity classifier:       {lat_id:.2f} ms")
print(f"\n  All models < 3 ms (single-sample CPU forward pass).")
print(f"  Suitable for real-time processing on a wearable device.")


LATENCY (for paper — text note, no figure)
  Baseline 1 (Stitched):       3.65 ms
  Baseline 2 (Output Fusion):  2.36 ms
  Feature Fusion (best combo): 5.30 ms
  + Identity classifier:       0.59 ms

  All models < 3 ms (single-sample CPU forward pass).
  Suitable for real-time processing on a wearable device.


In [49]:
print("\nAll outputs saved to:", OUT_DIR)
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name}")



All outputs saved to: /kaggle/working/paper_figures
  dataset_class_distribution_heldout.pdf
  dataset_class_distribution_heldout.png
  dataset_class_rate_heldout.pdf
  dataset_class_rate_heldout.png
  fig1_macro_f1_color.pdf
  fig1_macro_f1_color.png
  fig1_macro_f1_grayscale.pdf
  fig1_macro_f1_grayscale.png
  fig2_ff_per_class_color.pdf
  fig2_ff_per_class_color.png
  fig2_ff_per_class_grayscale.pdf
  fig2_ff_per_class_grayscale.png
  table_shuffle_summary.csv
  table_test_results.csv
  table_val_results.csv
